# Example: CellCognition Time-Lapse Phenotyping

A complete workflow for analyzing the temporal progression of phenotypes in time-lapse microscopy experiments, specifically tracing cell cycle and mitotic phases.

**Dataset:** [CellCognition Public Data](https://www.cellcognition.org/) (Time-lapse single-cell crops)

**Key Steps:**
1. **Metadata Mapping:** Load and logically order class labels to reflect temporal progression (e.g., mitotic phases).
2. **Feature Extraction:** Extract DINOv2 embeddings alongside classical morphological and intensity properties.
3. **Temporal Analysis:** Perform GMM clustering and visualize the phenospace to identify transitions between cell states.

In [ ]:
import torch
from phenome import PhenoMe, load_dinov2_model
import pandas as pd

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize PhenoMe and load model
pheno = PhenoMe(device=device, seed=42)
wrapper = load_dinov2_model("dinov2_vitg14_reg", device=device)

## 1. Paths & Configuration

Specify the paths to the time-lapse image dataset, corresponding masks, and the metadata CSV. By default, this notebook expects data in `data/CellCognition/`.

In [ ]:
IMAGE_DIR = "/datasets/CellCognition/Chromatin_microtubules/preprocessed_H2b_20x_MD_exp911/images"
MASKS_DIR = "/datasets/CellCognition/Chromatin_microtubules/preprocessed_H2b_20x_MD_exp911/masks"
METADATA_PATH = "/datasets/CellCognition/Chromatin_microtubules/preprocessed_H2b_20x_MD_exp911/metadata.csv"

## 2. Load Metadata & Mapping

We load the metadata and apply a mapping to ensure mitotic phases are ordered logically, reflecting the temporal progression of the cell cycle.

In [3]:
df_metadata = pd.read_csv(METADATA_PATH)
print(f"Loaded {len(df_metadata)} metadata entries.")

# Example mapping for H2B scheme (mitotic progression)
class_mapping = {
    "inter": "1_inter",
    "pro": "2_pro",
    "prometa": "3_prometa",
    "meta": "4_meta",
    "earlyana": "5_earlyana",
    "lateana": "6_lateana",
    "telo": "7_telo",
    "apo": "8_apo",
}
df_metadata["class"] = df_metadata["class"].map(class_mapping).fillna(df_metadata["class"])

# get only the rows in df_classes where the class is in get_lables
get_lables = ["1_inter", "2_pro", "3_prometa", "4_meta", "5_earlyana", "8_apo"]
df_metadata = df_metadata[df_metadata["class"].isin(get_lables)]

df_metadata.head()

Loaded 1011 metadata entries.


,filename,class,original_exp,rotation_angle
0,P0013_T00100_X0727_Y0787.png,8_apo,H2b_20x_MD_exp911,1.138922
1,P0013_T00100_X0736_Y0797.png,8_apo,H2b_20x_MD_exp911,-34.685462
2,P0013_T00100_X1320_Y0484.png,8_apo,H2b_20x_MD_exp911,-32.227924
3,P0013_T00100_X1321_Y0484.png,8_apo,H2b_20x_MD_exp911,-32.312173
4,P0013_T00100_X1322_Y0482.png,8_apo,H2b_20x_MD_exp911,-35.161960


In [4]:
pheno.transfer_metadata_to_properties("rotation_angle")

Transferred metadata columns ['rotation_angle'] to properties.


## 3. Data Discovery

Discover the single-cell crops and their associated masks, linking them with the mapped metadata for downstream analysis.

In [5]:
from phenome.utils import make_dataframe_metadata_fn

file_df = pheno.find_files(
    IMAGE_DIR,
    mask_dir=MASKS_DIR,
    metadata_fn=make_dataframe_metadata_fn(df_metadata)
)
file_df.head()

Found 1011 files total.
219 files have missing/failed metadata (drop).
Matched masks: 792/792 images (0 missing).
DataFrame: 792 files, 7 columns.


,file_path,filename,class,original_exp,rotation_angle,id,mask_path
0,/data3/agonzalez/datasets/CellCognition/Proces...,P0013_T00100_X0727_Y0787,8_apo,H2b_20x_MD_exp911,1.138922,P0013_T00100_X0727_Y0787,/data3/agonzalez/datasets/CellCognition/Proces...
1,/data3/agonzalez/datasets/CellCognition/Proces...,P0013_T00100_X0736_Y0797,8_apo,H2b_20x_MD_exp911,-34.685462,P0013_T00100_X0736_Y0797,/data3/agonzalez/datasets/CellCognition/Proces...
2,/data3/agonzalez/datasets/CellCognition/Proces...,P0013_T00100_X1320_Y0484,8_apo,H2b_20x_MD_exp911,-32.227924,P0013_T00100_X1320_Y0484,/data3/agonzalez/datasets/CellCognition/Proces...
3,/data3/agonzalez/datasets/CellCognition/Proces...,P0013_T00100_X1321_Y0484,8_apo,H2b_20x_MD_exp911,-32.312173,P0013_T00100_X1321_Y0484,/data3/agonzalez/datasets/CellCognition/Proces...
4,/data3/agonzalez/datasets/CellCognition/Proces...,P0013_T00100_X1322_Y0482,8_apo,H2b_20x_MD_exp911,-35.161960,P0013_T00100_X1322_Y0482,/data3/agonzalez/datasets/CellCognition/Proces...


## 4. Feature Extraction

We extract DINOv2 embeddings to capture deep phenotypic features and compute classical morphological properties (like area and intensity) to provide a comprehensive cell description.

> **Tip — Checkpoints & Resuming**:
> - **Save / Resume during extraction**: Pass `checkpoint_path="embeddings.h5"` to `process_images` and `checkpoint_path="properties.h5"` to `compute_properties`. Embeddings and properties are written incrementally to HDF5, saving RAM and enabling automatic resume if interrupted.
> - **Load precomputed checkpoints**: If checkpoints were already computed, skip feature extraction entirely by loading them with `pheno.load_embeddings(...)` and `pheno.load_properties(...)` (or `pheno.load_results(...)`).

In [ ]:
# Option A: Load precomputed checkpoints (skip extraction)
# pheno.load_embeddings("/path/to/cellcognition/.../embeddings.h5")
# pheno.load_properties("/path/to/cellcognition/.../properties.h5")

# Option B: Extract features (optionally add checkpoint_path to save/resume)
# Process images to extract embeddings
pheno.process_images(
    wrapper,
    batch_size=128,
    num_workers=4,
    channel_mode="split",
    resize_size=182,  # Target size for resizing images
    pad_size=100,  # Pad small images to 224x224 before resizing to 224x224
    l2_normalize_channels=False,
    # checkpoint_path="cellcognition_embeddings.h5",  # Save embeddings incrementally to HDF5
)

# Compute classical properties (area, intensity, etc.)
df_properties = pheno.compute_properties(
    property_preset="basic",
    n_jobs=8,
    # checkpoint_path="cellcognition_properties.h5",  # Save properties incrementally to HDF5
)

## 5. Visualization

We visualize the centroids of each mitotic phase in the phenospace using t-SNE, allowing us to observe the trajectory of phenotypic changes during the cell cycle.

In [7]:
# Visualize with t-SNE
pheno.plot_centroids(
    group_by="class",
    method="tsne",
)

## 6. Interactive Exploration

Launch the interactive explorer to visually inspect individual cells at different stages of the cell cycle and validate the identified phenotypic clusters.

In [ ]:
# In Google Colab, use pheno.create_colab_interactive_explorer() instead.
e= pheno.create_interactive_explorer()